Project Overview

Study: Metagenomics Analysis_16S rRNA gene sequencing of matched fecal, mucosal, and tumor microbiota from colorectal cancer patients

# Amplicon Sequencing Analysis of Colorectal Cancer Microbiome Samples

## Project Description

This notebook presents a reproducible QIIME 2 workflow for the analysis of 16S rRNA amplicon sequencing data from human colorectal cancer samples. The objective is to characterize microbial community composition and diversity across different sample types collected from colorectal cancer patients.

The analysis includes data import, quality assessment, denoising with DADA2, feature table generation, taxonomic classification, diversity analysis, and visualization.

## Computational Environment

- Operating System: Ubuntu (WSL2)
- Environment Manager: Miniconda
- Microbiome Analysis Platform: QIIME 2 (2026.4)
- Interactive Analysis: Jupyter Notebook
  
## Dataset Details

- BioProject: PRJNA1447725
- SRA Study: SRP694115
- Assay Type: Amplicon Sequencing
- Host Organism: Homo sapiens
- Sample Origin: South Korea (Daegu)
- Sequencing Platform: Illumina MiSeq
- Library Layout: Paired-end
- Organism: Human gut metagenome
- Total Samples: 18
- Submission: School of Medicine, Keimyung University, JeongWoo Hwang; 2026-04-16

## Sample Types

Samples were collected from colorectal cancer patients and include:

- Tumor tissue samples
- Adjacent mucosal tissue samples
- Fecal samples

Patient identifiers include P01–P06, with multiple sample types available for most patients.

## Analysis Goals

Sequencing data from human colorectal cancer samples. The objective is to characterize microbial community composition and diversity across different sample types collected from colorectal cancer patients.

1. Import and validate raw sequencing data.
2. Perform quality assessment of paired-end reads.
3. Denoise sequences using DADA2.
4. Generate amplicon sequence variants (ASVs).
5. Evaluate alpha and beta diversity.
6. Assign taxonomy using a reference database.
7. Identify microbial taxa associated with colorectal cancer sample types. 


## Project Directory

This notebook was developed and executed in a Linux (WSL Ubuntu) environment with all project files organized under the `~/metagenomics/` directory. Input datasets, intermediate files, and analysis outputs are stored in dedicated subfolders (metadata, sra, fastq, qiime2, results, taxonomy, annotation, etc.) to maintain an organized and reproducible workflow.

In [ ]:
## Creating Project Directories

Bioinformatics projects generate many intermediate and output files.

To keep the analysis organized, separate directories are created for:

- metadata: sample information
- sra: downloaded SRA files
- fastq: extracted sequencing reads
- qiime2: QIIME 2 artifacts and visualizations
- results: final outputs and figures

## Verify Current Working Directory

Before downloading files, verify that the notebook is operating within the project directory.

This ensures that all files are stored in the expected location.

In [4]:
!mkdir -p metadata
!mkdir -p sra
!mkdir -p fastq
!mkdir -p qiime2
!mkdir -p results
!pwd

/root/metagenomics


## Downloading Raw Sequencing Data AND Metadata

The raw paired-end FASTQ files and sample metadata corresponding to the colorectal cancer amplicon sequencing dataset will be downloaded from the NCBI Sequence Read Archive (SRA) using the SRA Toolkit.

The following SRA run accessions are included:
SRR38233223
SRR38233224
SRR38233225
SRR38233226
SRR38233227
SRR38233228
SRR38233229
SRR38233230
SRR38233231
SRR38233232
SRR38233233
SRR38233234
SRR38233235
SRR38233236
SRR38233237
SRR38233238
SRR38233239
SRR38233240


## Load Metadata Table

Metadata contains information about each sequencing sample.

Typical metadata fields include:

- Sample identifier
- SRA accession number
- Sample type
- Patient identifier
- Experimental group

Metadata is essential because QIIME 2 uses it for downstream statistical analysis and visualization.

In [8]:
import pandas as pd

metadata = pd.read_csv("metadata/SraRunTable.csv")
metadata.head()

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,LibraryLayout,LibrarySelection,LibrarySource,Organism,Platform,ReleaseDate,create_date,version,Sample Name,SRA Study
0,SRR38233223,AMPLICON,602,66362072,PRJNA1447725,SAMN57315478,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",38309070,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-12-17,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P04_feces,SRP694115
1,SRR38233224,AMPLICON,602,71240078,PRJNA1447725,SAMN57315477,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",40782819,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-10-11,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_tumor,SRP694115
2,SRR38233225,AMPLICON,602,71091986,PRJNA1447725,SAMN57315476,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",40980733,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-09-08,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_mucosa,SRP694115
3,SRR38233226,AMPLICON,602,66128496,PRJNA1447725,SAMN57315475,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",37987420,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2022-05-16,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_feces,SRP694115
4,SRR38233227,AMPLICON,602,57877484,PRJNA1447725,SAMN57315474,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",36009445,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-09-14,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P02_tumor,SRP694115


## Downloading Raw Sequencing Data

The metadata table contains SRA accession numbers for all samples.

In this step we download the raw sequencing data from NCBI SRA.

The download occurs in two stages:

### 1. prefetch

Downloads the original SRA archive file from NCBI.

Example:

SRR38233223.sra

### 2. fasterq-dump

Converts the SRA archive into FASTQ format that can be processed by QIIME 2.

Because this dataset contains paired-end sequencing reads, each sample will generate:

- Forward reads (_1.fastq)
- Reverse reads (_2.fastq)

These FASTQ files will be used as input for the QIIME 2 workflow.

In [2]:
!prefetch --version
!fasterq-dump --version


prefetch : 3.2.1


fasterq-dump : 3.2.1



In [16]:
runs = metadata["Run"].tolist()

for run in runs:
    print(run)

SRR38233223
SRR38233224
SRR38233225
SRR38233226
SRR38233227
SRR38233228
SRR38233229
SRR38233230
SRR38233231
SRR38233232
SRR38233233
SRR38233234
SRR38233235
SRR38233236
SRR38233237
SRR38233238
SRR38233239
SRR38233240


In [17]:
import subprocess

runs = metadata["Run"].tolist()

for run in runs:
    print(f"Downloading {run}...")
    subprocess.run(
        ["prefetch", run, "--output-directory", "sra"],
        check=True
    )

print("Download complete.")

2026-06-07T05:57:37 prefetch.3.2.1: 1) Resolving 'SRR38233223'...
2026-06-07T05:57:42 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-06-07T05:57:44 prefetch.3.2.1: 1) Downloading 'SRR38233223'...
2026-06-07T05:57:44 prefetch.3.2.1:  SRA Normalized Format file is being retrieved
2026-06-07T05:57:44 prefetch.3.2.1:  Downloading via HTTPS...
2026-06-07T05:59:10 prefetch.3.2.1:  HTTPS download succeed
2026-06-07T05:59:10 prefetch.3.2.1:  'SRR38233223' is valid: 38310989 bytes were streamed from 38297093
2026-06-07T05:59:10 prefetch.3.2.1: 1) 'SRR38233223' was downloaded successfully
2026-06-07T05:59:10 prefetch.3.2.1: 1) Resolving 'SRR38233223's dependencies...
2026-06-07T05:59:10 prefetch.3.2.1: 'SRR38233223' has 0 unresolved dependencies
2026-06-07T05:59:10 prefetch.3.2.1: 1) Resolving 'SRR38233224'...
2026-06-07T05:59:13 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base q

## Converting SRA Archives to FASTQ Files

The downloaded SRA archives cannot be analyzed directly in QIIME 2.

Therefore, each SRA file is converted into FASTQ format using `fasterq-dump`.

Because this dataset was generated using paired-end sequencing, each sample will produce:

- Forward reads (`_1.fastq`)
- Reverse reads (`_2.fastq`)

These FASTQ files will be imported into QIIME 2 for downstream processing.

## Test Conversion of One Sample

Before converting all 18 samples, we first convert a single SRA archive.

This verifies that:

- The downloaded file is valid.
- Paired-end FASTQ files are generated correctly.
- Output files are written to the expected directory.

Testing a single sample helps identify problems before processing the full dataset.

In [21]:
!fasterq-dump \
sra/SRR38233223/SRR38233223.sra \
-O fastq

spots read      : 110,236
reads read      : 220,472
reads written   : 220,472


In [22]:
!ls -lh fastq | head

total 159M
-rw-r--r-- 1 root root 80M Jun  7 07:21 SRR38233223_1.fastq
-rw-r--r-- 1 root root 80M Jun  7 07:21 SRR38233223_2.fastq


In [23]:
import subprocess

remaining_runs = metadata["Run"].tolist()[1:]

for run in remaining_runs:
    print(f"Converting {run}...")
    
    subprocess.run([
        "fasterq-dump",
        f"sra/{run}/{run}.sra",
        "-O",
        "fastq"
    ], check=True)

print("All conversions completed.")

Converting SRR38233224...


spots read      : 118,339
reads read      : 236,678
reads written   : 236,678


Converting SRR38233225...


spots read      : 118,093
reads read      : 236,186
reads written   : 236,186


Converting SRR38233226...


...

spots read      : 115,076
reads read      : 230,152
reads written   : 230,152


Converting SRR38233239...


spots read      : 138,223
reads read      : 276,446
reads written   : 276,446


Converting SRR38233240...
All conversions completed.


spots read      : 125,021
reads read      : 250,042
reads written   : 250,042


## Compressing FASTQ Files

QIIME 2 manifest import expects gzipped FASTQ files.

The FASTQ files generated by `fasterq-dump` are currently uncompressed. Therefore, they are compressed using gzip before import.

In [34]:
!gzip fastq/*.fastq

## Data Download and Conversion Summary

All sequencing data were successfully downloaded from the NCBI Sequence Read Archive (SRA).

Results:

- Samples downloaded: 18
- SRA files: 18
- FASTQ files generated: 36 (18 + 18)
- Library layout: Paired-end
- Total FASTQ size: ~3 GB

The dataset is now ready for quality assessment and import into QIIME 2.

Downstream Analysis

## Importing Paired-End FASTQ Files into QIIME 2

The raw sequencing reads are currently stored as FASTQ files.

QIIME 2 requires these files to be imported into its native artifact format (`.qza`).

A manifest file is used to associate each sample ID with its forward and reverse FASTQ files.

The resulting artifact will contain all demultiplexed paired-end sequences and will serve as the starting point for downstream quality assessment and DADA2 denoising.

NOTE: Why PairedEndFastqManifestPhred33V2?
It tells QIIME 2 that you are importing paired-end Illumina FASTQ files using a manifest file, with quality scores encoded as Phred+33. The quality scores in the FASTQ files use Phred+33 encoding (standard for modern Illumina data).

In [35]:
import pandas as pd
from pathlib import Path

manifest = pd.DataFrame({
    "sample-id": metadata["Run"],
    "forward-absolute-filepath": [
        str(Path.cwd() / "fastq" / f"{run}_1.fastq.gz")
        for run in metadata["Run"]
    ],
    "reverse-absolute-filepath": [
        str(Path.cwd() / "fastq" / f"{run}_2.fastq.gz")
        for run in metadata["Run"]
    ]
})

manifest.to_csv(
    "metadata/manifest.tsv",
    sep="\t",
    index=False
)

In [37]:
!qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path metadata/manifest.tsv \
  --output-path qiime2/paired-end-demux.qza \
  --input-format PairedEndFastqManifestPhred33V2

Imported metadata/manifest.tsv as PairedEndFastqManifestPhred33V2 to qiime2/paired-end-demux.qza


In [38]:
!qiime tools peek qiime2/paired-end-demux.qza

UUID:        d966194a-6c07-497b-9c09-6c58f8ce5e43
Type:        SampleData[PairedEndSequencesWithQuality]
Data format: SingleLanePerSamplePairedEndFastqDirFmt


## Generating a Demultiplexed Sequence Summary

This step generates a summary visualization of the imported paired-end sequencing data.

### Why is this step performed?

The summary provides an overview of the sequencing dataset, including:

- Number of sequences obtained for each sample.
- Distribution of sequencing depth across samples.
- Forward read quality scores.
- Reverse read quality scores.
- Read length distribution.

### Why is this important?

The quality score plots are used to determine appropriate filtering and truncation parameters for DADA2 denoising.

By examining where sequence quality begins to decline, we can choose suitable trimming positions that maximize data retention while minimizing sequencing errors.

# Visualizing the demultiplexed sequences quality control graphs
### Output

The resulting visualization file (`.qzv`) can be viewed in QIIME 2 View and will guide the selection of DADA2 parameters in the next step of the analysis.

In [39]:
!qiime demux summarize \
  --i-data qiime2/paired-end-demux.qza \
  --o-visualization qiime2/paired-end-demux.qzv

Saved Visualization to: qiime2/paired-end-demux.qzv


## Primer Removal and Read Trimming


After visualizing the demultiplexed sequences quality control graphs, All reads = exactly 301 nt (2nd–98th percentile identical). This is definitive evidence that primers were not trimmed.

So, primer sequences were removed from paired-end reads using the QIIME 2 Cutadapt plugin. Forward and reverse primers targeting the bacterial 16S rRNA gene were specified, and reads lacking the expected primer sequences were discarded to ensure that only correctly amplified target sequences were retained for downstream analysis. 

Again, summarize and visualize the trimmed .qza file by converting to .qzv

In [2]:
!qiime cutadapt trim-paired \
  --i-demultiplexed-sequences qiime2/paired-end-demux.qza \
  --p-front-f CCTACGGGNGGCWGCAG \
  --p-front-r GGACTACNVGGGTWTCTAAT \
  --p-discard-untrimmed \
  --o-trimmed-sequences qiime2/paired-end-demux-trimmed.qza \
  --verbose

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: cutadapt -u 0 --error-rate 0.1 --times 1 --overlap 3 --minimum-length 1 -q 0,0 --quality-base 33 --cores 1 -o /tmp/qiime2/root/processes/2100-1781499977.46@root/tmp/rachis-OutPath-9qrpm1lp/SRR38233223_0_L001_R1_001.fastq.gz -p /tmp/qiime2/root/processes/2100-1781499977.46@root/tmp/rachis-OutPath-9qrpm1lp/SRR38233223_18_L001_R2_001.fastq.gz --front CCTACGGGNGGCWGCAG -G GGACTACNVGGGTWTCTAAT -U 0 --discard-untrimmed /tmp/qiime2/root/data/d966194a-6c07-497b-9c09-6c58f8ce5e43/data/SRR38233223_0_L001_R1_001.fastq.gz /tmp/qiime2/root/data/d966194a-6c07-497b-9c09-6c58f8ce5e43/data/SRR38233223_18_L001_R2_001.fastq.gz

This is cutadapt 5.2 with Python 3.12.13
Command line parameters: -u 0 --error-rate 0.1 --times 1 --overlap 3 --minimum-length 1 -q 0,0 --qu

In [3]:
!qiime demux summarize \
  --i-data qiime2/paired-end-demux-trimmed.qza \
  --o-visualization qiime2/paired-end-demux-trimmed.qzv

Saved Visualization to: qiime2/paired-end-demux-trimmed.qzv


## Denoising Sequences with DADA2

DADA2 is used to denoise the paired-end sequencing reads and infer Amplicon Sequence Variants (ASVs).

Based on the quality profiles generated in the previous step, reads are truncated to remove low-quality regions while retaining sufficient sequence length for successful read merging.

Truncation parameters selected:

- Forward reads: 284 bp
- Reverse reads: 223 bp

DADA2 performs:
- Quality filtering
- Error-rate learning
- Sequence denoising
- Paired-end read merging
- Chimera removal

The output will include a feature table containing ASV counts per sample and representative ASV sequences.

In [5]:
!qiime dada2 denoise-paired \
  --i-demultiplexed-seqs qiime2/paired-end-demux-trimmed.qza \
  --p-trunc-len-f 284 \
  --p-trunc-len-r 223 \
  --o-table qiime2/table-dada2.qza \
  --o-representative-sequences qiime2/rep-seqs-dada2.qza \
  --o-denoising-stats qiime2/dada2-stats.qza \
  --o-base-transition-stats qiime2/base-transition-stats.qza

Saved FeatureTable[Frequency] to: qiime2/table-dada2.qza
Saved FeatureData[Sequence] to: qiime2/rep-seqs-dada2.qza
Saved SampleData[DADA2Stats] to: qiime2/dada2-stats.qza
Saved DADA2BaseTransitionStats to: qiime2/base-transition-stats.qza


## Creating a QIIME 2 Sample Metadata File

The original SRA metadata contains biological and technical information for each sample.

To enable sample annotation and downstream comparative analyses, the metadata are converted into a QIIME 2-compatible tab-separated metadata file.

The sample identifiers used in the metadata file must match the sample identifiers present in the QIIME 2 artifacts.

In [43]:
metadata.columns.tolist()

['Run',
 'Assay Type',
 'AvgSpotLen',
 'Bases',
 'BioProject',
 'BioSample',
 'BioSampleModel',
 'Bytes',
 'Center Name',
 'Collection_Date',
 'Consent',
 'DATASTORE filetype',
 'DATASTORE provider',
 'DATASTORE region',
 'env_broad_scale',
 'env_local_scale',
 'env_medium',
 'Experiment',
 'geo_loc_name_country',
 'geo_loc_name_country_continent',
 'geo_loc_name',
 'HOST',
 'Instrument',
 'lat_lon',
 'Library Name',
 'LibraryLayout',
 'LibrarySelection',
 'LibrarySource',
 'Organism',
 'Platform',
 'ReleaseDate',
 'create_date',
 'version',
 'Sample Name',
 'SRA Study']

In [14]:
import pandas as pd
import os

# Read original SRA metadata
df = pd.read_csv("/root/metagenomics/metadata/SraRunTable.csv")

# Create Patient_ID
df["Patient_ID"] = df["Sample Name"].str.extract(r"(P\d+)")

# Create Sample_Type
df["Sample_Type"] = df["Sample Name"].str.split("_").str[-1].str.capitalize()

# Select useful metadata columns
metadata = df[
    [
        "Run",
        "Sample Name",
        "Patient_ID",
        "Sample_Type",
        "HOST",
        "Organism",
        "geo_loc_name",
        "Collection_Date",
        "BioProject"
    ]
].copy()

# Rename Run column for QIIME2
metadata.rename(columns={"Run": "#SampleID"}, inplace=True)

# Save metadata file
metadata.to_csv(
    "/root/metagenomics/metadata/New_metadata.tsv",
    sep="\t",
    index=False
)

print("Metadata file saved successfully!")
print("\nColumns:")
print(metadata.columns.tolist())
print("\nShape:", metadata.shape)
print("\nPreview:")
print(metadata.head())

Metadata file saved successfully!

Columns:
['#SampleID', 'Sample Name', 'Patient_ID', 'Sample_Type', 'HOST', 'Organism', 'geo_loc_name', 'Collection_Date', 'BioProject']

Shape: (18, 9)

Preview:
     #SampleID Sample Name Patient_ID Sample_Type          HOST  \
0  SRR38233223   P04_feces        P04       Feces  Homo sapiens   
1  SRR38233224   P03_tumor        P03       Tumor  Homo sapiens   
2  SRR38233225  P03_mucosa        P03      Mucosa  Homo sapiens   
3  SRR38233226   P03_feces        P03       Feces  Homo sapiens   
4  SRR38233227   P02_tumor        P02       Tumor  Homo sapiens   

               Organism        geo_loc_name Collection_Date    BioProject  
0  human gut metagenome  South Korea: Daegu      2021-12-17  PRJNA1447725  
1  human gut metagenome  South Korea: Daegu      2021-10-11  PRJNA1447725  
2  human gut metagenome  South Korea: Daegu      2021-09-08  PRJNA1447725  
3  human gut metagenome  South Korea: Daegu      2022-05-16  PRJNA1447725  
4  human gut metagen

## Summarizing the DADA2 Feature Table

The DADA2 feature table contains the abundance of each Amplicon Sequence Variant (ASV) across all samples.

This step generates summary statistics describing:

- Number of detected ASVs.
- Total sequencing depth after denoising.
- Distribution of sequencing depth across samples.
- Feature frequencies and sample frequencies.

These summaries are used to evaluate the success of denoising and determine whether sufficient sequencing depth was retained for downstream analyses.

In [6]:
!qiime feature-table summarize \
  --i-table qiime2/table-dada2.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --o-feature-frequencies qiime2/feature-frequencies.qza \
  --o-sample-frequencies qiime2/sample-frequencies.qza \
  --o-summary qiime2/table-dada2.qzv

Saved ImmutableMetadata to: qiime2/feature-frequencies.qza
Saved ImmutableMetadata to: qiime2/sample-frequencies.qza
Saved Visualization to: qiime2/table-dada2.qzv


## Create rep-seqs-dada2.qzv

To inspect:

Number of ASVs
Sequence lengths
Actual nucleotide sequences
Later, associated taxonomy

What will we use these sequences for?
In a few steps, we'll classify them against a reference database (likely SILVA)

In [7]:
!qiime feature-table tabulate-seqs \
  --i-data qiime2/rep-seqs-dada2.qza \
  --o-visualization qiime2/rep-seqs-dada2.qzv

Saved Visualization to: qiime2/rep-seqs-dada2.qzv


## Evaluating DADA2 Read Retention

This step examines the number of reads retained after each stage of the DADA2 workflow.

The denoising statistics help identify where reads were lost during:

- Quality filtering
- Denoising
- Paired-end read merging
- Chimera removal

These metrics are used to evaluate whether the selected trimming parameters were appropriate and whether sufficient sequencing depth was retained for downstream analyses.

In [8]:
!qiime metadata tabulate \
  --m-input-file qiime2/dada2-stats.qza \
  --o-visualization qiime2/dada2-stats.qzv

Saved Visualization to: qiime2/dada2-stats.qzv


## Constructing a Phylogenetic Tree of ASVs

Many microbiome diversity metrics consider not only the presence and abundance of microorganisms but also their evolutionary relationships.

In this step, representative ASV sequences generated by DADA2 are aligned using MAFFT, a multiple sequence alignment algorithm. The aligned sequences are then used to construct a phylogenetic tree using FastTree.

The workflow generates:

- Multiple sequence alignment of ASVs
- Masked alignment with highly variable positions removed
- Unrooted phylogenetic tree
- Rooted phylogenetic tree

The rooted tree generated in this step will be used in subsequent diversity analyses.

In [9]:
!qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences qiime2/rep-seqs-dada2.qza \
  --o-alignment qiime2/aligned-rep-seqs.qza \
  --o-masked-alignment qiime2/masked-aligned-rep-seqs.qza \
  --o-tree qiime2/unrooted-tree.qza \
  --o-rooted-tree qiime2/rooted-tree.qza

Saved FeatureData[AlignedSequence] to: qiime2/aligned-rep-seqs.qza
Saved FeatureData[AlignedSequence] to: qiime2/masked-aligned-rep-seqs.qza
Saved Phylogeny[Unrooted] to: qiime2/unrooted-tree.qza
Saved Phylogeny[Rooted] to: qiime2/rooted-tree.qza


### Phylogenetic Tree Visualization (optional)

The rooted phylogenetic tree generated using MAFFT and FastTree was exported in Newick format and visualized using iTOL (Interactive Tree Of Life) for interactive exploration of ASV phylogenetic relationships.

The phylogenetic tree served as the basis for Faith's Phylogenetic Diversity and UniFrac distance calculations used in downstream diversity analyses.

In [12]:
!qiime tools export \
  --input-path qiime2/rooted-tree.qza \
  --output-path rooted_tree_export

Exported qiime2/rooted-tree.qza as NewickDirectoryFormat to directory rooted_tree_export


## Core Diversity Analysis

To enable meaningful comparisons between samples, the feature table is rarefied to a uniform sequencing depth.

Based on the DADA2 feature table summary, a sampling depth of 43000 reads was selected because it corresponds to the lowest sequencing depth observed among all samples. This allows all 18 samples to be retained for downstream analyses.

Using the rooted phylogenetic tree and rarefied feature table, QIIME 2 calculates a set of alpha-diversity and beta-diversity metrics.

### Alpha Diversity Metrics

These metrics describe diversity within individual samples:

- Observed Features (ASV richness)
- Shannon Diversity Index
- Faith's Phylogenetic Diversity
- Evenness

### Beta Diversity Metrics

These metrics compare microbial community composition between samples:

- Jaccard Distance
- Bray-Curtis Distance
- Unweighted UniFrac Distance
- Weighted UniFrac Distance

Principal Coordinate Analysis (PCoA) plots are also generated to visualize similarities and differences among samples.

The resulting diversity metrics will be used to investigate differences between tumor, mucosa, and fecal microbiomes.

Based on current feature-table statistics:
Max depth ≈ 43000 for alpha-rarefaction

## Alpha Rarefaction Analysis

Rarefaction analysis is performed to evaluate whether the sequencing depth is sufficient to capture the majority of microbial diversity present in the samples.

### Purpose
- Assess whether sequencing effort was adequate.
- Determine if diversity estimates have reached a plateau.
- Evaluate whether additional sequencing would likely reveal substantial new diversity.

A plateau in the rarefaction curves indicates that most microbial diversity has been captured and that sequencing depth is sufficient for downstream ecological analyses.

The maximum rarefaction depth is selected based on the observed sequencing depth distribution across samples while retaining as many samples as possible.

In [16]:
!qiime diversity alpha-rarefaction \
  --i-table qiime2/table-dada2.qza \
  --i-phylogeny qiime2/rooted-tree.qza \
  --p-max-depth 43000 \
  --m-metadata-file metadata/New_metadata.tsv \
  --o-visualization results/alpha-rarefaction.qzv

Saved Visualization to: results/alpha-rarefaction.qzv


## Alpha rarefaction interpretation

Rarefaction analysis was performed at a maximum sequencing depth of 43,000 reads, which corresponded to the minimum observed feature frequency across all 18 samples (range: 43,706–76,719 reads). All three alpha diversity metrics reached clear plateau/saturation by approximately 25,000–30,000 reads, confirming that the chosen sequencing depth was sufficient to capture the majority of microbial diversity present in each sample.

Across all three metrics, Feces samples consistently showed higher alpha diversity than Mucosa and Tumor samples. This is consistent with the known biology of the gut: the luminal compartment (feces) harbors a richer, more even microbial community compared to tissue-associated microbiomes.

The saturation of rarefaction curves at depths well below the maximum depth (43,000 reads) validates the adequacy of sequencing effort and confirms that additional sequencing would not substantially change diversity estimates.



## Beta Diversity Analysis

Beta diversity analysis was performed using Bray-Curtis, Jaccard, Weighted UniFrac, and Unweighted UniFrac distance metrics.

Principal Coordinate Analysis (PCoA) revealed clustering patterns associated with sample type.

Statistical significance was assessed using PERMANOVA (Permutational Multivariate Analysis of Variance).

In [23]:
!qiime diversity core-metrics-phylogenetic \
  --i-phylogeny qiime2/rooted-tree.qza \
  --i-table qiime2/table-dada2.qza \
  --p-sampling-depth 43000  \
  --m-metadata-file metadata/New_metadata.tsv \
  --output-dir core-metrics-results

Saved FeatureTable[Frequency] to: core-metrics-results/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/evenness_vector.qza
Saved DistanceMatrix to: core-metrics-results/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/jaccard_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/bray_curtis_distance_matrix.qza
Saved PCoAResults to: core-metrics-results/unweighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/weighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/jaccard_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/bray_curtis_pcoa_re

In [24]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/bray-curtis-permanova.qzv

Saved Visualization to: results/bray-curtis-permanova.qzv


In [25]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/jaccard_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/jaccard_distance_matrix-permanova.qzv

Saved Visualization to: results/jaccard_distance_matrix-permanova.qzv


In [26]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/unweighted_unifrac_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/unweighted_unifrac_distance_matrix-permanova.qzv

Saved Visualization to: results/unweighted_unifrac_distance_matrix-permanova.qzv


In [27]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/weighted_unifrac_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/weighted_unifrac_distance_matrix-permanova.qzv

Saved Visualization to: results/weighted_unifrac_distance_matrix-permanova.qzv


## Beta Diversity Analysis Interpretation

PERMANOVA results revealed a significant effect of sample type on microbial community composition when phylogenetic information was incorporated (Unweighted UniFrac: p=0.002; Weighted UniFrac: p=0.012). In contrast, metrics that do not consider phylogeny (Bray-Curtis and Jaccard) did not yield statistically significant results.

This pattern indicates that the compositional differences between Feces, Mucosa, and Tumor are primarily driven by differential abundances of phylogenetically distinct bacterial lineages, rather than simply by which taxa are present or absent. The higher pseudo-F statistic for Weighted UniFrac (2.97 vs. 1.96) further suggests that both the identity and relative abundance of phylogenetic lineages contribute to the observed community-level differences.

## Study Limitations

- This dataset contains matched fecal, mucosal, and tumor samples collected from the same six patients. Therefore, some statistical analyses (such as PERMANOVA) assume sample independence, which is not fully satisfied in this study.
- The results should be interpreted with this consideration, and future studies with larger cohorts and paired statistical approaches would provide stronger validation.

## Taxonomic Classification 


Taxonomic Classification (Current Status)

Taxonomic classification has not yet been completed for this project.

The initial plan was to classify Amplicon Sequence Variants (ASVs) generated by DADA2 using a pre-trained Greengenes classifier (gg-13-8-99-515-806-nb-classifier.qza). However, the classifier was trained using scikit-learn 1.4.2, while the current QIIME 2 2026.4 environment uses scikit-learn 1.7.1.

## Future Work (Updated)

Taxonomic assignment of ASVs.
Taxonomic composition analysis.
Differential abundance analysis between sample groups

## Taxonomic Classification Using Galaxy Europe

Training a custom Naive Bayes classifier locally in QIIME 2 (WSL Ubuntu) was not feasible due to memory limitations (8 GB RAM), resulting in repeated out-of-memory (OOM) termination during classifier training. To overcome this limitation, taxonomic classification was performed using **Galaxy Europe (QIIME 2 2026.1.0)** with the **`feature-classifier classify-consensus-vsearch`** plugin.

### Source

The SILVA 138.99 QIIME 2 reference artifacts were downloaded from the official QIIME 2 Data Resources repository and imported into Galaxy Europe for taxonomic classification.

### Inputs
- Representative sequences: `rep-seqs-dada2.qza`
- SILVA 138.99 reference sequences (341F–806R): `silva-138-99-seqs-341-806.qza`
- SILVA 138.99 taxonomy: `silva-138-99-tax.qza`

### Classification Parameters
- Percent identity: **0.90**
- Query coverage: **0.80**
- Maximum accepted hits: **10**
- Minimum consensus: **0.51**
- Strand: **both**
- Output unassigned sequences: **Yes**

### Outputs
- `classification.qza` – Final taxonomic assignments for all ASVs.
- `results.qza` – VSEARCH alignment results (BLAST6 format) used to derive consensus taxonomy.

This approach successfully generated taxonomic assignments while avoiding the computational limitations encountered during local classifier training.

In [9]:
cd ~/metagenomics/taxonomy

/root/metagenomics/taxonomy


In [10]:
ls -lh

total 484K
-rw-r--r-- 1 root root 196K Jul  1 10:58 classification.qza
-rw-r--r-- 1 root root 287K Jul  1 10:58 results.qza


In [11]:
!qiime tools peek results.qza

UUID:        da1d6cf3-8fe2-443d-bd01-f702ab794fe4
Type:        FeatureData[BLAST6]
Data format: BLAST6DirectoryFormat


In [12]:
!qiime tools peek classification.qza

UUID:        e3ef3c55-cde6-459b-a894-408b2095f74c
Type:        FeatureData[Taxonomy]
Data format: TSVTaxonomyDirectoryFormat


### Quality Control of Taxonomic Assignments

The taxonomy artifact was exported and inspected prior to downstream analysis. The **Consensus** value represents the fraction of accepted reference hits supporting the assigned taxonomy.

**Summary of consensus values**

| Consensus | Number of ASVs |
|-----------:|---------------:|
| 1.0 | 475 |
| 0.9 | 211 |
| 0.8 | 168 |
| 0.7 | 180 |
| 0.6 | 174 |
| Other intermediate values | 15 |

- Total classified ASVs: **1223**
- Unassigned ASVs: **86**
- Approximately **93.4%** of ASVs were successfully assigned taxonomy, while **6.6%** remained unassigned.
- Most classified ASVs exhibited high consensus (≥0.9), indicating reliable taxonomic assignments suitable for downstream taxonomic composition and differential abundance analyses.

In [13]:
!mkdir -p taxonomy_export

In [14]:
!qiime tools export \
    --input-path classification.qza \
    --output-path taxonomy_export

Exported classification.qza as TSVTaxonomyDirectoryFormat to directory taxonomy_export


In [16]:
!head -20 taxonomy_export/taxonomy.tsv

Feature ID	Taxon	Consensus
005a3b1fd45ceff8a65a343fab33f521	d__Bacteria; p__Firmicutes; c__Clostridia; o__Peptococcales; f__Peptococcaceae; g__Peptococcus; s__uncultured_bacterium	1.0
00a7069c61887549a552b1a92f4507fb	d__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Enterobacterales; f__Morganellaceae; g__Morganella; s__Morganella_morganii	1.0
00ffa432ea3445f71c4923b4cd5d954d	d__Bacteria; p__Firmicutes; c__Clostridia; o__Oscillospirales; f__Ruminococcaceae; g__Incertae_Sedis	0.9
0115d3e313fa9f2bd28c26b9572549e1	d__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Bacteroidales; f__Bacteroidaceae; g__Bacteroides; s__uncultured_bacterium	0.7
011dc8d28c65bbbbdf67e6e830f7022c	d__Bacteria; p__Firmicutes; c__Clostridia	1.0
013d2c7fb2317f174c808297db1922d5	Unassigned	1.0
01b3b2283ebeec4d9f7761b305522a3e	d__Bacteria; p__Firmicutes; c__Clostridia; o__Oscillospirales; f__Ruminococcaceae; g__Subdoligranulum; s__uncultured_bacterium	0.6
0227b8e982dd04e03eb3c6a52ac18db4	Unassigned	1.0
02490a1f

In [18]:
!cut -f3 taxonomy_export/taxonomy.tsv | sort | uniq -c

      2 0.571
    174 0.6
      3 0.667
    180 0.7
      1 0.714
      6 0.75
    168 0.8
      1 0.857
      2 0.875
    211 0.9
    475 1.0
      1 Consensus


In [20]:
!grep -c "Unassigned" taxonomy_export/taxonomy.tsv

86


In [ ]:
## Taxonomic Composition Visualization

Following successful taxonomic assignment, an interactive taxonomic composition plot was generated using the **QIIME 2 `taxa barplot`** plugin. The visualization integrates the feature table, taxonomic assignments, and sample metadata to display the relative abundance of microbial taxa across all samples.

### Inputs
- Feature table: `table-dada2.qza`
- Taxonomy: `classification.qza`
- Sample metadata: `New_metadata.tsv`

### Output
- `taxa-barplot.qzv`

In [23]:
!qiime taxa barplot \
  --i-table /root/metagenomics/qiime2/table-dada2.qza \
  --i-taxonomy /root/metagenomics/taxonomy/classification.qza \
  --m-metadata-file /root/metagenomics/metadata/New_metadata.tsv \
  --o-visualization /root/metagenomics/taxonomy/taxa-barplot.qzv

Saved Visualization to: /root/metagenomics/taxonomy/taxa-barplot.qzv


### Interpretation

The interactive bar plot displays the relative abundance of microbial taxa for each sample across multiple taxonomic levels (Domain to Species). Initial inspection showed:

- High microbial diversity across all colorectal samples.
- Considerable variation in community composition between individual samples.
- Detection of several gut-associated bacterial genera and species, including *Bacteroides*, *Blautia*, *Odoribacter*, *Sutterella*, *Desulfovibrio*, *Morganella*, and *Fusobacterium*.
- Numerous low-abundance taxa also contribute to overall community diversity.

## Differential Abundance Analysis Using ANCOM

Differential abundance analysis was performed using the **QIIME 2 Composition** plugin (**ANCOM**) to identify microbial taxa exhibiting significant differences in abundance among the three sample types (**Tumor**, **Mucosa**, and **Feces**).

### Workflow

1. Added a pseudocount to the DADA2 feature table using `qiime composition add-pseudocount`.
2. Generated a compositional feature table (`FeatureTable[Composition]`).
3. Performed ANCOM using the **Sample_Type** metadata column as the grouping variable.
4. Exported the ANCOM visualization (`ancom-sample-type.qzv`) for further inspection.
5. Annotated ANCOM results by merging feature IDs with the taxonomy assignments obtained from the SILVA 138.99 reference database.

### Input Files

- Feature table: `table-dada2.qza`
- Composition table: `composition-table.qza`
- Taxonomy: `classification.qza`
- Metadata: `New_metadata.tsv`

### Output Files

- `ancom-sample-type.qzv`
- `ancom.tsv`
- `ancom_annotated_results.tsv`

In [31]:
!head -10 ~/metagenomics/metadata/New_metadata.tsv

#SampleID	Sample Name	Patient_ID	Sample_Type	HOST	Organism	geo_loc_name	Collection_Date	BioProject
SRR38233223	P04_feces	P04	Feces	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-12-17	PRJNA1447725
SRR38233224	P03_tumor	P03	Tumor	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-10-11	PRJNA1447725
SRR38233225	P03_mucosa	P03	Mucosa	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-09-08	PRJNA1447725
SRR38233226	P03_feces	P03	Feces	Homo sapiens	human gut metagenome	South Korea: Daegu	2022-05-16	PRJNA1447725
SRR38233227	P02_tumor	P02	Tumor	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-09-14	PRJNA1447725
SRR38233228	P02_mucosa	P02	Mucosa	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-11-24	PRJNA1447725
SRR38233229	P02_feces	P02	Feces	Homo sapiens	human gut metagenome	South Korea: Daegu	2022-02-21	PRJNA1447725
SRR38233230	P01_tumor	P01	Tumor	Homo sapiens	human gut metagenome	South Korea: Daegu	2021-11-16	PRJNA1447725
SRR38233231	P06_tumor	P06

In [32]:
!qiime composition ancom --help

Usage: qiime composition ancom [OPTIONS]

  Apply Analysis of Composition of Microbiomes (ANCOM) to identify features
  that are differentially abundant across groups.

Inputs:
  --i-table ARTIFACT FeatureTable[Composition]
                         The feature table to be used for ANCOM computation.
                                                                    [required]
Parameters:
  --m-metadata-file METADATA
  --m-metadata-column COLUMN  MetadataColumn[Categorical]
                         The categorical sample metadata column to test for
                         differential abundance across.             [required]
  --p-transform-function TEXT Choices('sqrt', 'log', 'clr')
                         The method applied to transform feature values
                         before generating volcano plots.     [default: 'clr']
  --p-difference-function TEXT Choices('mean_difference', 'f_statistic')
                         The method applied to visualize fold difference in
      

In [33]:
!qiime composition add-pseudocount \
  --i-table /root/metagenomics/qiime2/table-dada2.qza \
  --o-composition-table /root/metagenomics/taxonomy/composition-table.qza

Saved FeatureTable[Composition] to: /root/metagenomics/taxonomy/composition-table.qza


In [34]:
!qiime composition ancom \
  --i-table /root/metagenomics/taxonomy/composition-table.qza \
  --m-metadata-file /root/metagenomics/metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --o-visualization /root/metagenomics/taxonomy/ancom-sample-type.qzv

Saved Visualization to: /root/metagenomics/taxonomy/ancom-sample-type.qzv


In [35]:
!mkdir -p /root/metagenomics/taxonomy/ancom_export

!qiime tools export \
    --input-path /root/metagenomics/taxonomy/ancom-sample-type.qzv \
    --output-path /root/metagenomics/taxonomy/ancom_export

Exported /root/metagenomics/taxonomy/ancom-sample-type.qzv as Visualization to directory /root/metagenomics/taxonomy/ancom_export


In [36]:
!ls -lh /root/metagenomics/taxonomy/ancom_export

total 356K
-rw-r--r-- 1 root root  49K Jul  2 09:18 ancom.tsv
drwxr-xr-x 2 root root 4.0K Jul  2 09:18 css
-rw-r--r-- 1 root root  65K Jul  2 09:18 data.tsv
-rw-r--r-- 1 root root 102K Jul  2 09:18 index.html
drwxr-xr-x 2 root root 4.0K Jul  2 09:18 js
drwxr-xr-x 2 root root 4.0K Jul  2 09:18 licenses
-rw-r--r-- 1 root root 116K Jul  2 09:18 percent-abundances.tsv
drwxr-xr-x 6 root root 4.0K Jul  2 09:18 q2templateassets


In [37]:
import pandas as pd

ancom = pd.read_csv(
    "/root/metagenomics/taxonomy/ancom_export/ancom.tsv",
    sep="\t"
)

ancom.head()

,Unnamed: 0,W,Signif
0,244302eb1874c008719ce609a61ba067,563,True
1,1a03eab87b42c9ce0c5101b738299f36,438,False
2,ad9a6343a04a31c1ec53dfd69875c5f0,41,False
3,8a2950244650b024368aa8174145ba33,28,False
4,f73409050ed46995d362beea6b884ddc,24,False


In [38]:
ancom.columns

Index(['Unnamed: 0', 'W', 'Signif'], dtype='object')

In [44]:
import pandas as pd

# Load ANCOM results
ancom = pd.read_csv(
    "/root/metagenomics/taxonomy/ancom_export/ancom.tsv",
    sep="\t"
)

# Rename the Feature ID column
ancom = ancom.rename(columns={"Unnamed: 0": "Feature ID"})

# Load taxonomy assignments
taxonomy = pd.read_csv(
    "/root/metagenomics/taxonomy/taxonomy_export/taxonomy.tsv",
    sep="\t"
)

# Merge ANCOM with taxonomy
merged = pd.merge(
    ancom,
    taxonomy,
    on="Feature ID",
    how="left"
)

# Sort by W statistic
merged = merged.sort_values("W", ascending=False)

# Save results
merged.to_csv(
    "/root/metagenomics/taxonomy/ancom_annotated_results.tsv",
    sep="\t",
    index=False
)

# Display the top 20 taxa
merged.head(20)

,Feature ID,W,Signif,Taxon,Consensus
0,244302eb1874c008719ce609a61ba067,563,True,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7
1,1a03eab87b42c9ce0c5101b738299f36,438,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,1.0
2,ad9a6343a04a31c1ec53dfd69875c5f0,41,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.9
3,8a2950244650b024368aa8174145ba33,28,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.6
4,f73409050ed46995d362beea6b884ddc,24,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7
5,38f9479d2f37d1036274704b09d2e3c5,12,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7
6,f7a1c2a7dc74eb24446faaa79f519ee6,9,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7
7,5c0644f23e7db04b7c8bd37098828313,7,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7
8,367da03193554afa419233cf58aa75c7,5,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.9
9,f360864823d0421bacdc21de7e63d06d,4,False,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.8


In [45]:
merged[["Feature ID","Taxon","Consensus","W","Signif"]].head(20)

,Feature ID,Taxon,Consensus,W,Signif
0,244302eb1874c008719ce609a61ba067,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7,563,True
1,1a03eab87b42c9ce0c5101b738299f36,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,1.0,438,False
2,ad9a6343a04a31c1ec53dfd69875c5f0,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.9,41,False
3,8a2950244650b024368aa8174145ba33,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.6,28,False
4,f73409050ed46995d362beea6b884ddc,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7,24,False
5,38f9479d2f37d1036274704b09d2e3c5,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7,12,False
6,f7a1c2a7dc74eb24446faaa79f519ee6,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7,9,False
7,5c0644f23e7db04b7c8bd37098828313,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.7,7,False
8,367da03193554afa419233cf58aa75c7,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.9,5,False
9,f360864823d0421bacdc21de7e63d06d,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.8,4,False


In [46]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

merged[["Feature ID", "Taxon", "Consensus", "W", "Signif"]].head(20)

,Feature ID,Taxon,Consensus,W,Signif
0,244302eb1874c008719ce609a61ba067,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Marvinbryantia; s__uncultured_bacterium,0.7,563,True
1,1a03eab87b42c9ce0c5101b738299f36,d__Bacteria; p__Firmicutes; c__Clostridia; o__Peptostreptococcales-Tissierellales; f__Peptostreptococcaceae; g__Peptostreptococcus,1.0,438,False
2,ad9a6343a04a31c1ec53dfd69875c5f0,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Lachnospiraceae_ND3007_group; s__uncultured_bacterium,0.9,41,False
3,8a2950244650b024368aa8174145ba33,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Blautia; s__uncultured_bacterium,0.6,28,False
4,f73409050ed46995d362beea6b884ddc,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Anaerostipes; s__uncultured_bacterium,0.7,24,False
5,38f9479d2f37d1036274704b09d2e3c5,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Blautia; s__uncultured_bacterium,0.7,12,False
6,f7a1c2a7dc74eb24446faaa79f519ee6,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Agathobacter,0.7,9,False
7,5c0644f23e7db04b7c8bd37098828313,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__[Ruminococcus]_torques_group; s__uncultured_bacterium,0.7,7,False
8,367da03193554afa419233cf58aa75c7,d__Bacteria; p__Firmicutes; c__Clostridia; o__Oscillospirales; f__Ruminococcaceae; g__Faecalibacterium; s__uncultured_bacterium,0.9,5,False
9,f360864823d0421bacdc21de7e63d06d,d__Bacteria; p__Firmicutes; c__Clostridia; o__Lachnospirales; f__Lachnospiraceae; g__Lachnospiraceae_NK4A136_group; s__uncultured_bacterium,0.8,4,False


### Results

ANCOM identified **one ASV as significantly differentially abundant** across the three sample groups.

| Feature | Taxonomy | W Statistic | Significant |
|---------|----------|------------:|:-----------:|
| 244302eb1874c008719ce609a61ba067 | *Marvinbryantia* (Family: Lachnospiraceae) | **563** |

The ASV was classified as:

```
Domain: Bacteria
Phylum: Firmicutes
Class: Clostridia
Order: Lachnospirales
Family: Lachnospiraceae
Genus: Marvinbryantia
Species: uncultured bacterium
```

Other taxa with relatively high W statistics included:

- *Peptostreptococcus* (W = 438)
- *Lachnospiraceae ND3007 group* (W = 41)
- *Blautia* (W = 28)
- *Anaerostipes* (W = 24)

However, these taxa did **not** meet the ANCOM significance criterion.

### Interpretation

The majority of ASVs were not identified as significantly different among Tumor, Mucosa, and Feces samples.

# Genus-Level Differential Abundance Analysis

## Objective

Following taxonomic assignment and ASV-level differential abundance analysis, the next objective is to investigate microbial community differences at the **genus level**. While ASVs provide the highest taxonomic resolution, genus-level analysis aggregates biologically related ASVs, improving interpretability and enabling comparison with findings reported in colorectal cancer microbiome studies.

---

## Why Perform Genus-Level Analysis?

The initial ANCOM analysis was performed on the original DADA2 ASV feature table (`table-dada2.qza`). Although this approach identifies differentially abundant individual sequence variants, multiple ASVs frequently belong to the same bacterial genus. Consequently, abundance differences may be distributed among several closely related ASVs, reducing statistical power and making biological interpretation more difficult.

Collapsing the feature table at the genus level combines all ASVs assigned to the same genus into a single taxonomic feature. This approach:

- Reduces data sparsity by combining related ASVs.
- Improves statistical power for detecting biologically meaningful differences.
- Produces results that are easier to interpret and compare across studies.
- Facilitates identification of bacterial genera associated with colorectal cancer.

## Generation of the Genus-Level Feature Table

The original DADA2 feature table (`table-dada2.qza`) was collapsed to **taxonomic Level 6 (Genus)** using the taxonomy assignments obtained from the SILVA 138.99 reference database.

### Input Files

- `table-dada2.qza` (ASV feature table)
- `classification.qza` (Taxonomic assignments)

### Output

- `genus-table.qza`

The resulting table contains the summed abundance of all ASVs belonging to the same bacterial genus for each sample.

## Calculation of Relative Genus Abundance

To facilitate visualization and comparison of microbial composition across samples, the collapsed genus count table was converted into relative abundances.

### Output

- `genus-relative.qza`

The relative abundance table expresses each genus as the proportion of the total microbial community within each sample and is useful for downstream visualization and exploratory analyses.

In [48]:
!qiime taxa collapse \
  --i-table /root/metagenomics/qiime2/table-dada2.qza \
  --i-taxonomy /root/metagenomics/taxonomy/classification.qza \
  --p-level 6 \
  --o-collapsed-table /root/metagenomics/taxonomy/genus-table.qza

Saved FeatureTable[Frequency] to: /root/metagenomics/taxonomy/genus-table.qza


In [49]:
!qiime feature-table relative-frequency \
  --i-table /root/metagenomics/taxonomy/genus-table.qza \
  --o-relative-frequency-table /root/metagenomics/taxonomy/genus-relative.qza

Saved FeatureTable[RelativeFrequency] to: /root/metagenomics/taxonomy/genus-relative.qza


## Next Step

Before performing ANCOM at the genus level, the genus abundance table (`genus-table.qza`) will be converted into a compositional feature table by adding a pseudocount. This transformation is required because ANCOM performs log-ratio based statistical analysis and cannot operate on feature tables containing zero counts.

In [50]:
!qiime composition add-pseudocount \
    --i-table /root/metagenomics/taxonomy/genus-table.qza \
    --o-composition-table /root/metagenomics/taxonomy/genus-composition.qza

Saved FeatureTable[Composition] to: /root/metagenomics/taxonomy/genus-composition.qza


In [51]:
!qiime tools peek /root/metagenomics/taxonomy/genus-composition.qza

UUID:        5ab9f783-9841-426b-89c6-a3fb0946effb
Type:        FeatureTable[Composition]
Data format: BIOMV210DirFmt


## Genus-Level Differential Abundance Analysis Using ANCOM

To identify bacterial **genera** that differ significantly in abundance among **Tumor**, **Mucosa**, and **Feces** samples.

### Why Genus-Level ANCOM?

The previous ANCOM analysis was performed using individual ASVs. Although ASVs provide the highest taxonomic resolution, closely related ASVs belonging to the same bacterial genus are analysed independently. This can reduce statistical power because biologically similar organisms are represented by multiple sequence variants.

### Input Files

`genus-composition.qza`- Compositional genus abundance table 
`New_metadata.tsv`- Sample metadata 
 Metadata column- `Sample_Type` 

### Output

- `genus-ancom.qzv`

The resulting visualization identifies bacterial genera exhibiting statistically significant differences in abundance among the three colorectal sample types.

In [52]:
!qiime composition ancom \
    --i-table /root/metagenomics/taxonomy/genus-composition.qza \
    --m-metadata-file /root/metagenomics/metadata/New_metadata.tsv \
    --m-metadata-column Sample_Type \
    --o-visualization /root/metagenomics/taxonomy/genus-ancom.qzv

Saved Visualization to: /root/metagenomics/taxonomy/genus-ancom.qzv


### Export the visualization
Following exactly the same workflow as before, but this time at the genus level.

In [53]:
!qiime tools export \
    --input-path genus-ancom.qzv \
    --output-path genus_ancom_export

Exported genus-ancom.qzv as Visualization to directory genus_ancom_export


As per geneus level annotation:

- Differential abundance analysis identified:
**Ruminococcus** (Family: *Ruminococcaceae*) as the only genus showing significant differences among **Tumor**, **Mucosa**, and **Feces** samples (W = 191). 

- The abundance distribution indicated that *Ruminococcus* was highly enriched in fecal samples and was present at very low abundance in both mucosal and tumor tissues.

## Export of the Genus-Level Feature Table

### Objective

To facilitate downstream statistical summaries and visualization outside the QIIME 2 framework, the genus-level feature table was exported into a standard BIOM format.

The exported table contains the abundance of each bacterial genus across all samples and serves as the basis for generating abundance summaries, identifying dominant taxa, and comparing microbial composition among Tumor, Mucosa, and Feces samples.

### Input

- `genus-table.qza`

### Output

- `feature-table.biom`

The exported BIOM table will be converted into a tabular format for downstream quantitative analysis and visualization.

In [1]:
!mkdir -p /root/metagenomics/taxonomy/genus_export

In [2]:
!qiime tools export \
    --input-path /root/metagenomics/taxonomy/genus-table.qza \
    --output-path /root/metagenomics/taxonomy/genus_export

Exported /root/metagenomics/taxonomy/genus-table.qza as BIOMV210DirFmt to directory /root/metagenomics/taxonomy/genus_export


## Export and Conversion of the Genus-Level Feature Table

The genus-level feature table was exported from the QIIME 2 artifact format and converted into a tab-separated values (TSV) file to enable downstream quantitative analyses in Python.


In [4]:
!biom convert \
    -i /root/metagenomics/taxonomy/genus_export/feature-table.biom \
    -o /root/metagenomics/taxonomy/genus_export/genus-table.tsv \
    --to-tsv

In [5]:
!head -20 /root/metagenomics/taxonomy/genus_export/genus-table.tsv

# Constructed from biom file
#OTU ID	SRR38233223	SRR38233224	SRR38233225	SRR38233226	SRR38233227	SRR38233228	SRR38233229	SRR38233230	SRR38233231	SRR38233232	SRR38233233	SRR38233234	SRR38233235	SRR38233236	SRR38233237	SRR38233238	SRR38233239	SRR38233240
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides	8319.0	24645.0	20227.0	16255.0	3056.0	6680.0	1155.0	17008.0	31783.0	27483.0	5169.0	44644.0	37721.0	1649.0	16536.0	11058.0	10964.0	692.0
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia-Shigella	8.0	2899.0	2328.0	11247.0	1256.0	374.0	0.0	100.0	17910.0	15960.0	193.0	8108.0	8546.0	25.0	4952.0	3363.0	56.0	32.0
d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_torques_group	1955.0	8775.0	4175.0	34.0	813.0	1252.0	2168.0	3569.0	13257.0	9848.0	4700.0	5155.0	3211.0	675.0	15520.0	5763.0	2822.0	229.0
d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospi

## Mean Genus Abundance by Sample Type

### Objective

The exported genus abundance table contains individual abundance values for each of the 18 samples. To facilitate biological interpretation, genus abundances were summarized according to sample type.

Comparing individual samples is often difficult due to biological variability. Therefore, samples were grouped into their respective clinical categories (**Tumor**, **Mucosa**, and **Feces**), and the mean abundance of each bacterial genus was calculated within each group.

This analysis provides an overview of the dominant bacterial genera characterizing each sample type and serves as the basis for comparative visualization and biological interpretation.

### Input

- `genus-table.tsv`
- `New_metadata.tsv`

### Output

- `mean_genus_abundance.csv`

The resulting table contains the average abundance of every bacterial genus across the three sample groups (Tumor, Mucosa, and Feces).

In [1]:
import pandas as pd

# Load genus abundance table
genus = pd.read_csv(
    "/root/metagenomics/taxonomy/genus_export/genus-table.tsv",
    sep="\t",
    skiprows=1
)

# Rename first column
genus.rename(columns={"#OTU ID": "Genus"}, inplace=True)

# Load metadata
meta = pd.read_csv(
    "/root/metagenomics/metadata/New_metadata.tsv",
    sep="\t"
)

meta.head()

,#SampleID,Sample Name,Patient_ID,Sample_Type,HOST,Organism,geo_loc_name,Collection_Date,BioProject
0,SRR38233223,P04_feces,P04,Feces,Homo sapiens,human gut metagenome,South Korea: Daegu,2021-12-17,PRJNA1447725
1,SRR38233224,P03_tumor,P03,Tumor,Homo sapiens,human gut metagenome,South Korea: Daegu,2021-10-11,PRJNA1447725
2,SRR38233225,P03_mucosa,P03,Mucosa,Homo sapiens,human gut metagenome,South Korea: Daegu,2021-09-08,PRJNA1447725
3,SRR38233226,P03_feces,P03,Feces,Homo sapiens,human gut metagenome,South Korea: Daegu,2022-05-16,PRJNA1447725
4,SRR38233227,P02_tumor,P02,Tumor,Homo sapiens,human gut metagenome,South Korea: Daegu,2021-09-14,PRJNA1447725


In [2]:
# Set genus names as index
genus = genus.set_index("Genus")

# Create mapping of sample -> sample type
sample_type = meta.set_index("#SampleID")["Sample_Type"]

# Average abundance by Sample Type
mean_abundance = genus.groupby(sample_type, axis=1).mean()

mean_abundance.head()

/tmp/ipykernel_1142/3896672441.py:8: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  mean_abundance = genus.groupby(sample_type, axis=1).mean()


Sample_Type,Feces,Mucosa,Tumor
Genus,,,
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides,5539.833333,19022.166667,22945.333333
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia-Shigella,1917.500000,5104.500000,5870.833333
d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667
d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Faecalibacterium,7696.833333,3982.666667,3450.000000
d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667


In [3]:
mean_abundance.to_csv(
    "/root/metagenomics/taxonomy/mean_genus_abundance.csv"
)

mean_abundance.head(20)

Sample_Type,Feces,Mucosa,Tumor
Genus,,,
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides,5539.833333,19022.166667,22945.333333
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia-Shigella,1917.500000,5104.500000,5870.833333
d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667
d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Faecalibacterium,7696.833333,3982.666667,3450.000000
d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667
d__Bacteria;p__Firmicutes;c__Bacilli;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Holdemanella,485.666667,1161.166667,1770.000000
d__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Clostridiaceae;g__Clostridium_sensu_stricto_1,567.000000,2800.833333,1139.500000
d__Bacteria;p__Fusobacteriota;c__Fusobacteriia;o__Fusobacteriales;f__Fusobacteriaceae;g__Fusobacterium,41.000000,2798.666667,2333.333333
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella,4934.333333,4757.333333,2704.333333


## Preparation of Genus Names for Visualization

### Objective

The genus abundance table contains complete SILVA taxonomic lineages. To improve readability in downstream visualizations, the taxonomy strings were simplified by extracting only the bacterial genus names.

Displaying the full taxonomic lineage on plots is difficult to interpret and occupies unnecessary space. Extracting the genus names produces cleaner tables and figures while preserving the biological information required for comparative analysis of Tumor, Mucosa, and Feces samples.

### Output

A simplified abundance table containing bacterial genus names and their corresponding mean abundances across the three sample types was generated for visualization and interpretation.

In [4]:
import pandas as pd

# Load mean abundance table
mean = pd.read_csv(
    "/root/metagenomics/taxonomy/mean_genus_abundance.csv"
)

# Extract genus names from taxonomy strings
mean["Genus"] = mean["Genus"].str.extract(r"g__([^;]+)$")

# Replace missing genus names
mean["Genus"] = mean["Genus"].fillna("Unclassified")

# Display first rows
mean.head()

,Genus,Feces,Mucosa,Tumor
0,Bacteroides,5539.833333,19022.166667,22945.333333
1,Escherichia-Shigella,1917.500000,5104.500000,5870.833333
2,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667
3,Faecalibacterium,7696.833333,3982.666667,3450.000000
4,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667


## Identification of the Top 20 Most Abundant Genera

### Objective

To identify the dominant bacterial genera present across all colorectal cancer samples.

Microbiome datasets typically contain hundreds of bacterial genera, many of which are present at very low abundance. Focusing on the most abundant genera simplifies interpretation and highlights taxa that contribute most to the microbial community.

The overall mean abundance of each genus was calculated across the three sample groups (Feces, Mucosa, and Tumor), and the twenty most abundant genera were selected for downstream visualization and biological interpretation.

### Input

- `mean_genus_abundance.csv`

### Output

- `top20_genera.csv`

This table contains the twenty most abundant bacterial genera ranked according to their overall mean abundance across all samples.

In [5]:
# Calculate overall mean abundance across all sample types
mean["Overall"] = mean[["Feces", "Mucosa", "Tumor"]].mean(axis=1)

# Select the 20 most abundant genera
top20 = mean.sort_values("Overall", ascending=False).head(20)

# Display the results
top20

,Genus,Feces,Mucosa,Tumor,Overall
0,Bacteroides,5539.833333,19022.166667,22945.333333,15835.777778
3,Faecalibacterium,7696.833333,3982.666667,3450.000000,5043.166667
2,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778
1,Escherichia-Shigella,1917.500000,5104.500000,5870.833333,4297.611111
8,Prevotella,4934.333333,4757.333333,2704.333333,4132.000000
9,Klebsiella,1374.000000,2398.666667,2538.666667,2103.777778
7,Fusobacterium,41.000000,2798.666667,2333.333333,1724.333333
4,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667,1525.611111
6,Clostridium_sensu_stricto_1,567.000000,2800.833333,1139.500000,1502.444444
11,Blautia,3565.333333,302.000000,389.000000,1418.777778


In [7]:
top20 = (
    mean.sort_values("Overall", ascending=False)
        .head(20)
        .reset_index(drop=True)
)

top20.index = top20.index + 1

top20

,Genus,Feces,Mucosa,Tumor,Overall
1,Bacteroides,5539.833333,19022.166667,22945.333333,15835.777778
2,Faecalibacterium,7696.833333,3982.666667,3450.000000,5043.166667
3,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778
4,Escherichia-Shigella,1917.500000,5104.500000,5870.833333,4297.611111
5,Prevotella,4934.333333,4757.333333,2704.333333,4132.000000
6,Klebsiella,1374.000000,2398.666667,2538.666667,2103.777778
7,Fusobacterium,41.000000,2798.666667,2333.333333,1724.333333
8,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667,1525.611111
9,Clostridium_sensu_stricto_1,567.000000,2800.833333,1139.500000,1502.444444
10,Blautia,3565.333333,302.000000,389.000000,1418.777778


## Ranking of Dominant Bacterial Genera

### Objective

The bacterial genera were ranked according to their overall mean abundance across Feces, Mucosa, and Tumor samples.

Ranking genera by their average abundance provides an overview of the dominant members of the microbial community before examining sample type-specific differences. This step identifies the genera contributing most to the overall microbiome and serves as the basis for downstream comparative analyses and visualizations.

## Identification of the Top 10 Genera Within Each Sample Type

### Objective

To determine the dominant bacterial genera present in **Feces**, **Mucosa**, and **Tumor** samples individually.

Although the overall abundance ranking identifies the dominant genera across the complete dataset, microbial communities differ substantially between sample types. Ranking genera separately for each sample group enables identification of taxa that are specifically enriched within fecal, mucosal, or tumor-associated microbiomes.

This analysis facilitates biological interpretation by highlighting characteristic microbial signatures associated with each sample type.

### Input

- `mean_genus_abundance.csv`

### Outputs

- `top10_feces.csv`
- `top10_mucosa.csv`
- `top10_tumor.csv`

These tables summarize the ten most abundant bacterial genera within each sample group and provide the basis for comparative visualization and interpretation.

In [8]:
# Top 10 genera in each sample type

top10_feces = mean.sort_values("Feces", ascending=False)[["Genus","Feces"]].head(10).reset_index(drop=True)
top10_mucosa = mean.sort_values("Mucosa", ascending=False)[["Genus","Mucosa"]].head(10).reset_index(drop=True)
top10_tumor = mean.sort_values("Tumor", ascending=False)[["Genus","Tumor"]].head(10).reset_index(drop=True)

# Display the tables
print("Top 10 Genera - Feces")
display(top10_feces)

print("\nTop 10 Genera - Mucosa")
display(top10_mucosa)

print("\nTop 10 Genera - Tumor")
display(top10_tumor)

Top 10 Genera - Feces


,Genus,Feces
0,Faecalibacterium,7696.833333
1,Bacteroides,5539.833333
2,Prevotella,4934.333333
3,Blautia,3565.333333
4,Subdoligranulum,2019.000000
5,Escherichia-Shigella,1917.500000
6,Akkermansia,1916.000000
7,[Ruminococcus]_torques_group,1626.833333
8,Ruminococcus,1565.500000
9,Streptococcus,1484.666667



Top 10 Genera - Mucosa


,Genus,Mucosa
0,Bacteroides,19022.166667
1,Escherichia-Shigella,5104.500000
2,Prevotella,4757.333333
3,[Ruminococcus]_torques_group,4511.833333
4,Faecalibacterium,3982.666667
5,[Ruminococcus]_gnavus_group,2863.166667
6,Clostridium_sensu_stricto_1,2800.833333
7,Fusobacterium,2798.666667
8,Klebsiella,2398.666667
9,Alloprevotella,1689.333333



Top 10 Genera - Tumor


,Genus,Tumor
0,Bacteroides,22945.333333
1,[Ruminococcus]_torques_group,7848.166667
2,Escherichia-Shigella,5870.833333
3,Faecalibacterium,3450.000000
4,Prevotella,2704.333333
5,Klebsiella,2538.666667
6,Fusobacterium,2333.333333
7,Holdemanella,1770.000000
8,Haemophilus,1738.833333
9,Streptococcus,1707.333333


In [9]:
top10_feces.to_csv("/root/metagenomics/taxonomy/top10_feces.csv", index=False)

top10_mucosa.to_csv("/root/metagenomics/taxonomy/top10_mucosa.csv", index=False)

top10_tumor.to_csv("/root/metagenomics/taxonomy/top10_tumor.csv", index=False)

m### Key Findings

Several bacterial genera, including **Bacteroides**, **Faecalibacterium**, **Escherichia-Shigella**, **Prevotella**, and the **Ruminococcus torques group**, were consistently among the most abundant taxa across all sample types, representing the core microbial community of the dataset.

## Integration of Dominant Genera Across Sample Types

### Input

- `top10_feces.csv`
- `top10_mucosa.csv`
- `top10_tumor.csv`

### Output

- `dominant_genera_all_groups.csv`

The resulting table contains the union of dominant bacterial genera across all three sample types and their corresponding mean abundances.

In [10]:
import pandas as pd

# Create the union of genera from the three Top 10 tables
selected_genera = sorted(
    set(top10_feces["Genus"])
    | set(top10_mucosa["Genus"])
    | set(top10_tumor["Genus"])
)

# Keep only these genera from the mean abundance table
combined = mean[mean["Genus"].isin(selected_genera)].copy()

# Sort by overall abundance
combined = combined.sort_values("Overall", ascending=False)

combined

,Genus,Feces,Mucosa,Tumor,Overall
0,Bacteroides,5539.833333,19022.166667,22945.333333,15835.777778
3,Faecalibacterium,7696.833333,3982.666667,3450.000000,5043.166667
2,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778
1,Escherichia-Shigella,1917.500000,5104.500000,5870.833333,4297.611111
8,Prevotella,4934.333333,4757.333333,2704.333333,4132.000000
9,Klebsiella,1374.000000,2398.666667,2538.666667,2103.777778
7,Fusobacterium,41.000000,2798.666667,2333.333333,1724.333333
4,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667,1525.611111
6,Clostridium_sensu_stricto_1,567.000000,2800.833333,1139.500000,1502.444444
11,Blautia,3565.333333,302.000000,389.000000,1418.777778


In [11]:
combined.to_csv(
    "/root/metagenomics/taxonomy/dominant_genera_all_groups.csv",
    index=False
)

In [ ]:
## Visualization of top genera 

### Figure: Grouped Bar Plot of the Top 20 Dominant Bacterial Genera

### Objective

To compare the mean abundance of the twenty most abundant bacterial genera across Feces, Mucosa, and Tumor samples.

The grouped bar plot enables direct comparison of genus abundances among the three sample types, facilitating identification of genera that are enriched or depleted in specific microbial communities.

This visualization complements the statistical analyses by providing an intuitive overview of microbial composition differences associated with colorectal cancer.

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load Top 20 genera
top20 = pd.read_csv("/root/metagenomics/taxonomy/top20_genera.csv")

# Positions for grouped bars
x = np.arange(len(top20))
width = 0.25

plt.figure(figsize=(18,8))

plt.bar(x - width,
        top20["Feces"],
        width,
        label="Feces")

plt.bar(x,
        top20["Mucosa"],
        width,
        label="Mucosa")

plt.bar(x + width,
        top20["Tumor"],
        width,
        label="Tumor")

plt.xticks(
    x,
    top20["Genus"],
    rotation=75,
    ha="right",
    fontsize=10
)

plt.ylabel("Mean abundance", fontsize=12)
plt.xlabel("Bacterial Genus", fontsize=12)
plt.title("Top 20 Dominant Bacterial Genera Across Sample Types", fontsize=14)

plt.legend()

plt.tight_layout()

plt.savefig(
    "/root/metagenomics/taxonomy/top20_grouped_barplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

/tmp/ipykernel_1142/3439327023.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figure: Heatmap of the Top 20 Dominant Bacterial Genera

In [13]:
import pandas as pd
import matplotlib.pyplot as plt

# Load Top 20 table
top20 = pd.read_csv("/root/metagenomics/taxonomy/top20_genera.csv")

# Prepare matrix
heat = top20.set_index("Genus")[["Feces","Mucosa","Tumor"]]

plt.figure(figsize=(9,10))

plt.imshow(heat, aspect="auto", cmap="viridis")

plt.colorbar(label="Mean abundance")

plt.xticks(
    range(3),
    heat.columns,
    fontsize=12
)

plt.yticks(
    range(len(heat.index)),
    heat.index,
    fontsize=9
)

plt.title(
    "Heatmap of Top 20 Dominant Bacterial Genera",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    "/root/metagenomics/taxonomy/top20_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

/tmp/ipykernel_1142/3698263346.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Figures: Top 10 Dominant Genera Within Each Sample Type

In [14]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_top10(csv_file, sample_name, color):

    df = pd.read_csv(csv_file)

    plt.figure(figsize=(10,6))

    plt.barh(df["Genus"], df.iloc[:,1])

    plt.xlabel("Mean abundance")
    plt.ylabel("Genus")

    plt.title(f"Top 10 Dominant Genera - {sample_name}")

    plt.gca().invert_yaxis()

    plt.tight_layout()

    outfile = f"/root/metagenomics/taxonomy/top10_{sample_name.lower()}_barplot.png"

    plt.savefig(outfile, dpi=300)

    plt.show()


plot_top10(
    "/root/metagenomics/taxonomy/top10_feces.csv",
    "Feces",
    "steelblue"
)

plot_top10(
    "/root/metagenomics/taxonomy/top10_mucosa.csv",
    "Mucosa",
    "forestgreen"
)

plot_top10(
    "/root/metagenomics/taxonomy/top10_tumor.csv",
    "Tumor",
    "firebrick"
)

/tmp/ipykernel_1142/2758384608.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1142/2758384608.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1142/2758384608.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Annotation of Dominant Genera Using the Disbiome Database

### Objective

To annotate the dominant bacterial genera identified in this study with previously reported colorectal cancer (CRC) associations using the Disbiome database.

### Method

The colorectal cancer dataset was retrieved from the Disbiome database in JSON format and imported into Python. Genus names from the Top 20 abundance table were standardized to match the Disbiome nomenclature (e.g., resolving SILVA/QIIME2-specific genus names). The normalized genera were then matched against curated CRC-associated microorganisms to identify previously reported associations.

### Output

- `disbiome_crc.json` – Colorectal cancer records retrieved from Disbiome
- `top20_disbiome_annotation.csv` – Annotated table containing dominant genera, CRC evidence, reported direction of association (Elevated/Reduced), sample type, experimental method, and supporting publication identifiers.

### Significance

This annotation step integrates experimental microbiome results with curated disease-associated microbial evidence, enabling biological interpretation of the dominant genera observed in the colorectal cancer cohort.

In [1]:
!mkdir /root/metagenomics/annotation/

In [2]:
import requests
import json
import os

url = "https://disbiome.ugent.be:8080/experiment?search=colorectal%20cancer"

response = requests.get(
    url,
    headers={"Accept": "application/json"}
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

os.makedirs("/root/metagenomics/annotation", exist_ok=True)

with open("/root/metagenomics/annotation/disbiome_crc.json", "w") as f:
    f.write(response.text)

print("Saved as /root/metagenomics/annotation/disbiome_crc.json")

Status code: 200
Content-Type: None
Saved as /root/metagenomics/annotation/disbiome_crc.json


In [3]:
with open("/root/metagenomics/annotation/disbiome_crc.json") as f:
    text = f.read()

print(text[:1000])

[{"experiment_id":221,"methodological_detail_id":null,"qualitative_outcome":"Elevated","disease_id":22,"meddra_level":"preferred_term","meddra_id":10061451,"location_id":10,"disease_name":"Colorectal cancer","organism_id":44,"organism_name":"Prevotella","organism_ncbi_id":59823,"subject_value":"-1.0400","control_value":"-1.4000","ratio":null,"method_name":"qPCR","sample_name":"Faeces","host_type":"Human","host_id":21,"publication_id":21,"control_name":"Healthy control","response_name":"Log10","response_unit":"Log10","hittypes":[{"origin":"disease","original_terms":["Colorectal cancer","Colorectal cancer"]},{"origin":"meddra","original_terms":["Colorectal cancer","Colorectal carcinoma","Recurrent N-ras mutation-positive colorectal carcinoma","Colorectal cancer NOS","Colorectal neoplasms malignant","Colorectal cancer NOS","Colorectal cancer"]}]}, {"experiment_id":220,"methodological_detail_id":null,"qualitative_outcome":"Elevated","disease_id":22,"meddra_level":"preferred_term","meddra_i

In [4]:
import json
import pandas as pd

with open("/root/metagenomics/annotation/disbiome_crc.json") as f:
    disbiome = json.load(f)

print("Number of experiments:", len(disbiome))

pd.DataFrame(disbiome).head()

Number of experiments: 388


,experiment_id,methodological_detail_id,qualitative_outcome,disease_id,meddra_level,meddra_id,location_id,disease_name,organism_id,organism_name,...,ratio,method_name,sample_name,host_type,host_id,publication_id,control_name,response_name,response_unit,hittypes
0,221,NaN,Elevated,22,preferred_term,10061451,10,Colorectal cancer,44,Prevotella,...,None,qPCR,Faeces,Human,21.0,21,Healthy control,Log10,Log10,"[{'origin': 'disease', 'original_terms': ['Col..."
1,220,NaN,Elevated,22,preferred_term,10061451,10,Colorectal cancer,36,Bacteroides,...,None,qPCR,Faeces,Human,21.0,21,Healthy control,Log10,Log10,"[{'origin': 'disease', 'original_terms': ['Col..."
2,538,NaN,Reduced,22,preferred_term,10061451,18,Colorectal cancer,290,Anoxybacillus,...,None,16S rRNA sequencing,Tissue biopsie,Human,68.0,64,Same person (healthy tissue),None,None,"[{'origin': 'disease', 'original_terms': ['Col..."
3,537,NaN,Reduced,22,preferred_term,10061451,18,Colorectal cancer,289,Microbacterium,...,None,16S rRNA sequencing,Tissue biopsie,Human,68.0,64,Same person (healthy tissue),None,None,"[{'origin': 'disease', 'original_terms': ['Col..."
4,536,NaN,Elevated,22,preferred_term,10061451,18,Colorectal cancer,14,Roseburia,...,None,16S rRNA sequencing,Tissue biopsie,Human,68.0,64,Same person (healthy tissue),None,None,"[{'origin': 'disease', 'original_terms': ['Col..."


In [7]:
import json
import pandas as pd

# Load the Disbiome JSON
with open("/root/metagenomics/annotation/disbiome_crc.json", "r") as f:
    disbiome = json.load(f)

# Convert to DataFrame
df_dis = pd.DataFrame(disbiome)

print("Number of records:", len(df_dis))
print(df_dis.head())

Number of records: 388
   experiment_id  methodological_detail_id qualitative_outcome  disease_id  \
0            221                       NaN            Elevated          22   
1            220                       NaN            Elevated          22   
2            538                       NaN             Reduced          22   
3            537                       NaN             Reduced          22   
4            536                       NaN            Elevated          22   

     meddra_level  meddra_id  location_id       disease_name  organism_id  \
0  preferred_term   10061451           10  Colorectal cancer           44   
1  preferred_term   10061451           10  Colorectal cancer           36   
2  preferred_term   10061451           18  Colorectal cancer          290   
3  preferred_term   10061451           18  Colorectal cancer          289   
4  preferred_term   10061451           18  Colorectal cancer           14   

    organism_name  ...  ratio          method

In [8]:
top20 = pd.read_csv("/root/metagenomics/taxonomy/top20_genera.csv")

top20.head()

,Genus,Feces,Mucosa,Tumor,Overall
0,Bacteroides,5539.833333,19022.166667,22945.333333,15835.777778
1,Faecalibacterium,7696.833333,3982.666667,3450.000000,5043.166667
2,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778
3,Escherichia-Shigella,1917.500000,5104.500000,5870.833333,4297.611111
4,Prevotella,4934.333333,4757.333333,2704.333333,4132.000000


In [9]:
import re

def normalize_genus(genus):
    """
    Convert SILVA/QIIME genus names to standard genus names
    for comparison with Disbiome.
    """

    # Remove square brackets
    genus = genus.replace("[", "").replace("]", "")

    # Remove everything after first underscore
    genus = genus.split("_")[0]

    # Handle hyphenated genera
    if genus == "Escherichia-Shigella":
        genus = "Escherichia"

    # Handle Ruminococcus groups
    if genus == "Ruminococcus":
        return "Ruminococcus"

    return genus


top20["Genus_Normalized"] = top20["Genus"].apply(normalize_genus)

top20[["Genus","Genus_Normalized"]].head(20)

,Genus,Genus_Normalized
0,Bacteroides,Bacteroides
1,Faecalibacterium,Faecalibacterium
2,[Ruminococcus]_torques_group,Ruminococcus
3,Escherichia-Shigella,Escherichia
4,Prevotella,Prevotella
5,Klebsiella,Klebsiella
6,Fusobacterium,Fusobacterium
7,[Ruminococcus]_gnavus_group,Ruminococcus
8,Clostridium_sensu_stricto_1,Clostridium
9,Blautia,Blautia


In [10]:
df_dis["Genus_Normalized"] = (
    df_dis["organism_name"]
    .str.replace("[","", regex=False)
    .str.replace("]","", regex=False)
)

In [11]:
matches = top20.merge(
    df_dis[["Genus_Normalized"]].drop_duplicates(),
    on="Genus_Normalized",
    how="left",
    indicator=True
)

matches[["Genus","Genus_Normalized","_merge"]]

,Genus,Genus_Normalized,_merge
0,Bacteroides,Bacteroides,both
1,Faecalibacterium,Faecalibacterium,both
2,[Ruminococcus]_torques_group,Ruminococcus,both
3,Escherichia-Shigella,Escherichia,both
4,Prevotella,Prevotella,both
5,Klebsiella,Klebsiella,both
6,Fusobacterium,Fusobacterium,both
7,[Ruminococcus]_gnavus_group,Ruminococcus,both
8,Clostridium_sensu_stricto_1,Clostridium,left_only
9,Blautia,Blautia,both


In [12]:
summary = []

for genus in top20["Genus_Normalized"]:

    subset = df_dis[df_dis["Genus_Normalized"] == genus]

    if subset.empty:
        summary.append({
            "Genus_Normalized": genus,
            "CRC_Studies": 0,
            "Elevated": 0,
            "Reduced": 0,
            "Sample_Types": "None",
            "Methods": "None",
            "Publications": 0
        })
    else:
        summary.append({

            "Genus_Normalized": genus,

            "CRC_Studies": len(subset),

            "Elevated":
                (subset["qualitative_outcome"]=="Elevated").sum(),

            "Reduced":
                (subset["qualitative_outcome"]=="Reduced").sum(),

            "Sample_Types":
                ", ".join(sorted(subset["sample_name"].dropna().unique())),

            "Methods":
                ", ".join(sorted(subset["method_name"].dropna().unique())),

            "Publications":
                subset["publication_id"].nunique()

        })

annotation = pd.DataFrame(summary)

annotation.head()

,Genus_Normalized,CRC_Studies,Elevated,Reduced,Sample_Types,Methods,Publications
0,Bacteroides,9,3,6,"Faeces, Tissue biopsie","16S rDNA pyrosequencing, 16S rRNA pyrosequenci...",9
1,Faecalibacterium,5,0,5,"Faeces, Rectal swab, Tissue biopsie","16S rDNA pyrosequencing, 16S rRNA sequencing",4
2,Ruminococcus,5,3,2,"Faeces, Tissue biopsie","16S rRNA sequencing, Metagenomic sequencing",4
3,Escherichia,2,2,0,"Faeces, Tissue biopsie","16S rRNA pyrosequencing, 16S rRNA sequencing",2
4,Prevotella,5,4,1,"Faeces, Oral swab, Tissue biopsie","16S rRNA sequencing, qPCR",5


In [13]:
final_annotation = top20.merge(
    annotation,
    on="Genus_Normalized",
    how="left"
)

final_annotation

,Genus,Feces,Mucosa,Tumor,Overall,Genus_Normalized,CRC_Studies,Elevated,Reduced,Sample_Types,Methods,Publications
0,Bacteroides,5539.833333,19022.166667,22945.333333,15835.777778,Bacteroides,9,3,6,"Faeces, Tissue biopsie","16S rDNA pyrosequencing, 16S rRNA pyrosequenci...",9
1,Faecalibacterium,7696.833333,3982.666667,3450.000000,5043.166667,Faecalibacterium,5,0,5,"Faeces, Rectal swab, Tissue biopsie","16S rDNA pyrosequencing, 16S rRNA sequencing",4
2,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778,Ruminococcus,5,3,2,"Faeces, Tissue biopsie","16S rRNA sequencing, Metagenomic sequencing",4
3,[Ruminococcus]_torques_group,1626.833333,4511.833333,7848.166667,4662.277778,Ruminococcus,5,3,2,"Faeces, Tissue biopsie","16S rRNA sequencing, Metagenomic sequencing",4
4,Escherichia-Shigella,1917.500000,5104.500000,5870.833333,4297.611111,Escherichia,2,2,0,"Faeces, Tissue biopsie","16S rRNA pyrosequencing, 16S rRNA sequencing",2
5,Prevotella,4934.333333,4757.333333,2704.333333,4132.000000,Prevotella,5,4,1,"Faeces, Oral swab, Tissue biopsie","16S rRNA sequencing, qPCR",5
6,Klebsiella,1374.000000,2398.666667,2538.666667,2103.777778,Klebsiella,3,1,2,"Rectal swab, Tissue biopsie",16S rRNA sequencing,2
7,Fusobacterium,41.000000,2798.666667,2333.333333,1724.333333,Fusobacterium,18,18,0,"Faeces, Rectal swab, Tissue biopsie","16S rDNA pyrosequencing, 16S rRNA sequencing, ...",15
8,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667,1525.611111,Ruminococcus,5,3,2,"Faeces, Tissue biopsie","16S rRNA sequencing, Metagenomic sequencing",4
9,[Ruminococcus]_gnavus_group,106.000000,2863.166667,1607.666667,1525.611111,Ruminococcus,5,3,2,"Faeces, Tissue biopsie","16S rRNA sequencing, Metagenomic sequencing",4


In [14]:
final_annotation.to_csv(
    "/root/metagenomics/taxonomy/top20_disbiome_annotation.csv",
    index=False
)

### Results - ## Biological Annotation of Dominant Genera

The majority of the dominant genera identified in the colorectal cancer cohort were supported by previous curated evidence in Disbiome. Notably, *Fusobacterium*, *Bacteroides*, *Faecalibacterium*, *Prevotella*, and *Streptococcus* were represented by multiple independent colorectal cancer studies. These annotations provide biological context for the observed microbial community composition and support the relevance of the identified taxa.

Genera without matching records in the downloaded Disbiome colorectal cancer dataset were retained in the results and are considered as having no curated evidence in the current database release rather than being considered unrelated to colorectal cancer.

# Summary of Results

## Objective

To characterize and compare the gut microbial communities present in matched fecal, mucosal, and tumor samples from colorectal cancer patients using 16S rRNA gene amplicon sequencing data.

## Analysis Workflow Completed

The following analyses were successfully completed:

1. Raw sequence quality assessment and preprocessing.
2. Denoising and ASV generation using the DADA2 pipeline.
3. Taxonomic classification using the SILVA reference database with the Galaxy Europe_QIIME 2 VSEARCH consensus classifier.
4. Interactive taxonomic composition visualization using QIIME 2 Taxa Barplot.
5. Taxonomic collapse to the genus level and calculation of relative abundances.
6. Differential abundance analysis using ANCOM at both feature (ASV) and genus levels.
7. Identification of the dominant bacterial genera across fecal, mucosal, and tumor samples.
8. Comparative abundance analysis of the Top 20 genera and Top 10 genera within each sample type.
9. Generation of visualizations, including bar plots and heatmaps of dominant genera.
10. Biological annotation of dominant genera using the Disbiome colorectal cancer database.

## Major Findings

- Taxonomic profiling identified distinct microbial communities across fecal, mucosal, and tumor samples.
- The dominant genera included **Bacteroides, Faecalibacterium, Prevotella, Escherichia-Shigella, Ruminococcus, Fusobacterium, Klebsiella, Blautia, Akkermansia,** and **Roseburia**.
- ANCOM identified taxa showing significant differences in abundance among the three sample types, indicating microbial shifts associated with the colorectal cancer microenvironment.
- Comparative abundance analysis demonstrated variation in dominant genera between feces, mucosa, and tumor tissues.
- Biological annotation using the Disbiome database showed that the majority of dominant genera have previously reported associations with colorectal cancer.
- Particularly strong literature support was observed for **Fusobacterium**, **Bacteroides**, **Faecalibacterium**, **Streptococcus**, and **Prevotella**, while a small number of genera had no curated associations in the current Disbiome colorectal cancer dataset.

## Outputs Generated

- DADA2 feature table and representative sequences
- Taxonomic classification using Galaxy Europe_QIIME 2 VSEARCH consensus classifier
- Interactive taxonomy bar plots (.qzv)
- Genus-level abundance tables
- Relative abundance tables
- ANCOM differential abundance results
- Top 20 dominant genera table
- Top 10 genera tables for feces, mucosa, and tumor
- Publication-quality bar plots and heatmaps
- Disbiome-annotated dominant genera table (`top20_disbiome_annotation.csv`)

## Conclusion

This workflow successfully established an end-to-end QIIME 2 microbiome analysis pipeline for colorectal cancer 16S rRNA sequencing data. The integration of taxonomic profiling, differential abundance analysis, visualization, and external biological annotation enabled comprehensive characterization of microbial communities and provided biologically meaningful insights into taxa associated with colorectal cancer.